In [13]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from PIL import Image
from io import BytesIO
import io
import ast
import matplotlib.pyplot as plt
import base64
import json
import os
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import gc
# gc.collect() 
# torch.cuda.empty_cache() 
# torch.cuda.ipc_collect()

In [14]:
def jsons_para_dataframe(pasta):
    dados = []

    for arquivo in os.listdir(pasta):
        if arquivo.endswith(".json"):
            caminho = os.path.join(pasta, arquivo)
            
            with open(caminho, "r", encoding="utf-8") as f:
                conteudo = json.load(f)
            
            if isinstance(conteudo, dict):
                conteudo["_arquivo"] = arquivo
                dados.append(conteudo)
            else:
                dados.append({
                    "_arquivo": arquivo,
                    "conteudo": conteudo
                })

    df = pd.DataFrame(dados)
    return df


pasta = "/home/nicolas/VQA-RAD-Matiaz/Proj_Mosca/resultsmark"
df = jsons_para_dataframe(pasta)
print(f"{len(df)} artigos carregados.")
df.head()

105 artigos carregados.


,level,biblio,body_text,figures_and_tables,references,_arquivo
0,paragraph,{'title': 'Edge effects on the blowfly fauna (...,"[{'id': 'p_d5223977', 'text': 'The Tijuca Nati...","[{'id': 'fig_0', 'label': '1', 'head': 'Figure...","[{'id': 'b1', 'target': 'b0', 'title': 'Associ...",Gadelha et al. - 2015 - Edge effects on the bl...
1,paragraph,{'title': 'Larvicidal efficacy of nitenpyram o...,"[{'id': 'p_fb060aaf', 'text': 'Cochliomyia hom...","[{'id': 'fig_0', 'label': '', 'head': '*', 'ty...","[{'id': 'b1', 'target': 'b0', 'title': 'Inquir...",Correia TR et al. - 2010 - Larvicidal efficacy...
2,paragraph,{'title': 'Hipertratamento de miíase decorrent...,"[{'id': 'p_49d11ab7', 'text': 'COSTA, et al.',...","[{'id': 'fig_0', 'label': '1', 'head': 'Figura...","[{'id': 'b2', 'target': 'b1', 'journal': 'Rev....",Costa FS et al. - 2012 - Hipertratamento de mi...
3,paragraph,{'title': 'Infl uência Climática e Antrópica n...,"[{'id': 'p_66923403', 'text': 'Os califorídeos...","[{'id': 'fig_0', 'label': '', 'head': '', 'typ...","[{'id': 'b1', 'target': 'b0', 'title': 'Combin...",Ferraz AC et al. - 2010 - Influência climática...
4,paragraph,{'title': 'Inquérito sobre os casos de miíase ...,"[{'id': 'p_27de7ffa', 'text': 'Um inquérito so...",[],"[{'id': 'b1', 'target': 'b0', 'title': 'The me...",Cramer-Ribeiro et al. - 2002 - Inquérito sobre...


In [15]:
df.iloc[0].biblio

{'title': 'Edge effects on the blowfly fauna (Diptera, Calliphoridae) of the Tijuca National Park, Rio de Janeiro, Brazil',
 'authors': ['B Gadelha', 'A Ribeiro', 'V Aguiar', 'C Mello-Patiu'],
 'doi': '10.1590/1519-6984.05614',
 'hash': '90A0C167DD3D0060FF82F96BD8451B07',
 'publication_date': '2015-11-10',
 'publication_year': 2015,
 'publisher': 'FapUNIFESP (SciELO)',
 'journal': 'Brazilian Journal of Biology',
 'abstract': [{'id': 0,
   'text': 'In this contribution we examine the diversity, abundance and species richness of Calliphoridae in the Tijuca National Park, Rio de Janeiro, RJ. Specimens were collected monthly between September 2009 and August 2010, using sardines as bait, in three points: A, on the forest edge (22°56\'846"S 43°17\'496"W), B, 700m from it (22°57\'073"S 43°17\'832"W) and C, 1,200m into the forest (22°57\'321"S 43°18\'031"W), evaluating the indicator species at each point and the anthropogenic influences and abiotic factors that determine species distribution.

In [16]:
rf = pd.read_json("/home/nicolas/VQA-RAD-Matiaz/Proj_Mosca/outputs/Qwen_Qwen3.5-9B_Sumarization.jsonl", lines=True)
rf

,id,model,sumarization
0,0,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
1,1,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""M"",\n ..."
2,2,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""M"",\n ..."
3,3,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
4,4,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""M"",\n ..."
...,...,...,...
98,100,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
99,101,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
100,102,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": null,\n ..."
101,103,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": null,\n ..."


In [17]:
rf.iloc[1].sumarization

'{\n  "hosts": [\n    {\n      "sex": "M",\n      "age": 9,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    },\n    {\n      "sex": "M",\n      "age": 10,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    },\n    {\n      "sex": "M",\n      "age": 11,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    },\n    {\n      "sex": "M",\n      "age": 12,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    },\n    {\n      "sex": "M",\n      "age": 13,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    },\n    {\n      "sex": "M",\n      "age": 14,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    },\n    {\n      "sex": "F",\n      "age": 6,\n      "species": "Canis familiaris",\n      "type": "CompanionAnimal"\n    }\n  ],\n  "vectors": [\n    {\n      "species": "Cochliomyia hominivorax",\n      "myiasis_type": "Primary",\n      "synonyms"

In [18]:
def extrair_texto(body_text):
    """
    Recebe a coluna body_text (lista de dicts com chave 'text')
    e retorna uma string única com todos os parágrafos concatenados.
    """
    if isinstance(body_text, list):
        paragrafos = [p["text"] for p in body_text if isinstance(p, dict) and "text" in p]
        return " ".join(paragrafos).strip()
    
    # fallback: caso já seja string
    return str(body_text).strip()


# Teste com o primeiro artigo
sample = df.iloc[0]
texto_extraido = extrair_texto(sample["body_text"])

print(f"Título: {sample['biblio'].get('title', 'sem título') if isinstance(sample['biblio'], dict) else ''}")
print(f"\nPrimeiros 500 caracteres do texto extraído:")
print(texto_extraido[:500])
print(f"\nTotal de caracteres: {len(texto_extraido)}")

Título: Edge effects on the blowfly fauna (Diptera, Calliphoridae) of the Tijuca National Park, Rio de Janeiro, Brazil

Primeiros 500 caracteres do texto extraído:
The Tijuca National Park (PARNATijuca), with an area of 35.51 km 2 , is located on the metropolitan area of the city of Rio de Janeiro. The vegetation in the reserve is part of the Atlantic Forest biome, which is known for its high rates of endemism (ca 50%). Unfortunately, for many years the Atlantic Forest has been subjected to a growing and irreversible process of fragmentation; the original vegetation has been extensively destroyed, and humans occupy most areas surrounding the few fragments 

Total de caracteres: 24073


In [19]:
df["texto_extraido"] = df["body_text"].apply(extrair_texto)
df = df.drop(columns=["body_text"])

In [20]:
def load_model(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_buffers=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    return model, tokenizer

In [21]:
def build_messages(texto):
    return [
        {
    "role": "system",
    "content": [{"type": "text", "text": (
        "You are an assistant specialized in summarizing scientific texts about parasitology and infectious diseases. "
        """Extract the requested information from the text and return your response as a JSON object matching this exact structure:

{
  "hospedeiros": [
    {
      "sexo": "M",
      "idade": "35 anos",
      "especie": "Homo sapiens",
      "tipo": "HUMANO"
    },
    {
      "sexo": "F",
      "idade": "4 anos",
      "especie": "Canis lupus familiaris",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "M",
      "idade": "2 anos",
      "especie": "Bos taurus",
      "tipo": "ANIMAL_PRODUCAO"
    },
    {
      "sexo": "F",
      "idade": "1 ano",
      "especie": "Didelphis marsupialis",
      "tipo": "ANIMAL_SELVAGEM"
    }
  ],
  "vetores": [
    {
      "nome_cientifico": "Cochliomyia hominivorax",
      "nome_comum_json": ["bicheira", "screwworm"]
    }
  ],
  "parasitos": [
    {
      "nome_cientifico": "Cochliomyia hominivorax",
      "ordem_animal": "Diptera",
      "estagio_vida": "larva",
      "nome_doenca": "Miíase Cavitária"
    }
  ],
  "doencas": [
    {
      "nome_cientifico": "Myiasis",
      "nome_comum_json": ["miíase", "bicheira", "miíase cavitária"]
    }
  ],
  "locais": [
    {
      "uf": "MA",
      "cidade": "São Luís",
      "bairro": "Cohama",
      "longitude": null,
      "latitude": null
    }
  ],
  "ocorrencias": [
    {
      "data": "2026-03-15",
      "parte_corpo": "Braço",
      "tratamento": "Remoção larval",
      "desfecho": "Curado"
    }
  ]
}

Rules:
- Return ONLY the JSON object — no explanations, no markdown fences, no extra text.
- Every field must be present. Use null for any value that cannot be found in the text.
- Do not infer or hallucinate values; extract only information explicitly stated in the source text.
- Dates must follow ISO 8601 format (YYYY-MM-DD).
- Deduplicate hospedeiros, vetores, parasitos, doencas, locais and fontes arrays — each unique entity appears only once.
- tipo field for hospedeiros must be exactly one of: HUMANO, ANIMAL_COMPANHIA, ANIMAL_PRODUCAO, ANIMAL_SELVAGEM.
- idade should be a string preserving the original unit (e.g. '3 meses', '2 anos', 'adulto').
- vetores are arthropods or organisms that mechanically or biologically transmit the parasito (e.g. flies, ticks, mosquitoes). Never classify a fly species as a hospedeiro.
- parasitos are the organisms that cause the doenca (e.g. fly larvae causing myiasis). A parasito must always reference a doenca via nome_doenca matching a doenca entry's nome_cientifico or nome_comum_json.
- doencas represent the clinical condition caused by the parasito (e.g. Myiasis). Extract both scientific and common names when available.
- A vetor and a parasito can share the same species name when the organism acts both as vector and causative agent (e.g. Cochliomyia hominivorax larvae cause the disease AND the adult fly is the vector).
- tipo_fonte should be one of: Artigo Científico, Relato de Caso, Livro, Tese, Dissertação, Relatório, Site, Outro."""
    )}]
},
{
    "role": "user",
    "content": [{"type": "text", "text": texto}],
}
        ]

In [22]:
def run_inference(model, tokenizer, messages):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False   # desativa o modo de raciocínio interno
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,  # suficiente para o JSON completo
            do_sample=False,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
    output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Garante que tokens de thinking residuais não contaminem o JSON
    output = re.sub(r"<think>.*?</think>", "", output, flags=re.DOTALL).strip()

    del inputs, generated_ids
    gc.collect()
    torch.cuda.empty_cache()

    return output

In [23]:
def run_experiment(
    df,
    model_name,
    output_dir="outputs"
):
    not_processed = [6,13,21]
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(
        output_dir,
        model_name.replace("/", "_") + "_Sumarization3.jsonl"
    )

    # ==========================
    # Ler ids já processados
    # ==========================
    processed_ids = set()

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    processed_ids.add(record["id"])
                except:
                    pass

    print(f"{len(processed_ids)} exemplos já processados.")

    model, tokenizer = load_model(model_name)

    with open(output_path, "a", encoding="utf-8") as f:

        for idx, row in df.iterrows():

            if idx in not_processed:
                continue
            # ==========================
            # Pular se já foi respondido
            # ==========================
            if idx in processed_ids:
                print(f"[SKIP] {model_name} | {idx}")
                continue

            texto_extraido = row["texto_extraido"]

            messages = build_messages(
                texto=texto_extraido
            )

            sumarization = run_inference(
                model=model,
                tokenizer=tokenizer,
                messages=messages
)

            record = {
                "id": idx,
                "model": model_name,
                "sumarization": sumarization
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

            print(f"[{model_name} | {idx}] {sumarization}")

            gc.collect()
            torch.cuda.empty_cache()

    return output_path

In [24]:
run_experiment(
        df=df,
        model_name="Qwen/Qwen3.5-9B"
    )

0 exemplos já processados.


Loading weights: 100%|██████████| 427/427 [00:01<00:00, 357.72it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 0] {
  "hospedeiros": [
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Chrysomya megacephala",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Chrysomya albiceps",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Chrysomya macellaria",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Chrysomya hominivorax",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Chrysomya idioidea",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Chrysomya putoria",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Laneella nigripes",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto"

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 1] {
  "hospedeiros": [
    {
      "sexo": "M",
      "idade": "9 meses",
      "especie": "Canis lupus familiaris",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "M",
      "idade": "6 anos",
      "especie": "Canis lupus familiaris",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "1 ano",
      "especie": "Canis lupus familiaris",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "M",
      "idade": "adulto",
      "especie": "Homo sapiens",
      "tipo": "HUMANO"
    },
    {
      "sexo": "M",
      "idade": "adulto",
      "especie": "Bos taurus",
      "tipo": "ANIMAL_PRODUCAO"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Canis lupus familiaris",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "M",
      "idade": "adulto",
      "especie": "Felis catus",
      "tipo": "ANIMAL_COMPANHIA"
    },
    {
      "sexo": "F",
      "idade": "adulto",
      "especie": "Fe

...

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 102] {
  "hospedeiros": [
    {
      "sexo": null,
      "idade": null,
      "especie": "Sus scrofa",
      "tipo": "ANIMAL_PRODUCAO"
    },
    {
      "sexo": null,
      "idade": null,
      "especie": "Bos taurus",
      "tipo": "ANIMAL_PRODUCAO"
    },
    {
      "sexo": null,
      "idade": null,
      "especie": "Homo sapiens",
      "tipo": "HUMANO"
    }
  ],
  "vetores": [
    {
      "nome_cientifico": "Chloroprocta idioidea",
      "nome_comum_json": [
        "Chloroprocta idioidea"
      ]
    },
    {
      "nome_cientifico": "Chrysomya albiceps",
      "nome_comum_json": [
        "Chrysomya albiceps"
      ]
    },
    {
      "nome_cientifico": "Mesembrinella bicolor",
      "nome_comum_json": [
        "Mesembrinella bicolor"
      ]
    },
    {
      "nome_cientifico": "Chrysomya putoria",
      "nome_comum_json": [
        "Chrysomya putoria"
      ]
    },
    {
      "nome_cientifico": "Chrysomya megacephala",
      "nome_comum_json": [
   

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 103] {
  "hospedeiros": [
    {
      "sexo": null,
      "idade": null,
      "especie": "Sus scrofa domesticus",
      "tipo": "ANIMAL_PRODUCAO"
    },
    {
      "sexo": null,
      "idade": null,
      "especie": "Homo sapiens",
      "tipo": "HUMANO"
    },
    {
      "sexo": null,
      "idade": null,
      "especie": "Bos taurus",
      "tipo": "ANIMAL_PRODUCAO"
    }
  ],
  "vetores": [
    {
      "nome_cientifico": "Cochliomyia hominivorax",
      "nome_comum_json": [
        "bicheira",
        "screwworm"
      ]
    },
    {
      "nome_cientifico": "Chloroprocta idioidea",
      "nome_comum_json": [
        "bicheira"
      ]
    },
    {
      "nome_cientifico": "Chrysomya megacephala",
      "nome_comum_json": [
        "bicheira"
      ]
    },
    {
      "nome_cientifico": "Chrysomya putoria",
      "nome_comum_json": [
        "bicheira"
      ]
    },
    {
      "nome_cientifico": "Hemilucilia segmentaria",
      "nome_comum_json": [
        "

'outputs/Qwen_Qwen3.5-9B_Sumarization3.jsonl'